# Cassandra Data Modeling – Telecom Usage Database

## Project
Telecom Usage Event Platform with Apache Cassandra

## Objective
Designing a query-driven data model for storing and retrieving
telecom usage events using Apache Cassandra.

## Scope
- Define application query patterns.
- Identify suitable partition and clustering keys.
- Validate primary key uniqueness.
- Design Cassandra table structures.

## Tools
- Python
- Pandas
- Apache Cassandra

In [2]:
import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)
print("Environment ready.")

Pandas version: 3.0.5
Environment ready.


In [4]:
DATA_PATH = Path("../data/raw/MPD_sample_synthetic_ken_3000subs.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully
Shape: (1072608, 7)
Columns: ['msisdn', 'datetime', 'cell_id', 'latitude', 'longitude', 'data_type', 'service']


In [5]:
subscriber_time_groups = (
    df.groupby(["msisdn", "datetime"])
      .size()
)

duplicate_groups = subscriber_time_groups[
    subscriber_time_groups > 1
]

print("===== Subscriber + Timestamp Uniqueness =====")
print("Total subscriber-timestamp groups:",
      len(subscriber_time_groups))

print("Duplicate groups:",
      len(duplicate_groups))

print("Records in duplicate groups:",
      duplicate_groups.sum())

print("Maximum events at the same timestamp:",
      subscriber_time_groups.max())

===== Subscriber + Timestamp Uniqueness =====
Total subscriber-timestamp groups: 1072608
Duplicate groups: 0
Records in duplicate groups: 0
Maximum events at the same timestamp: 1


In [7]:
df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%Y-%m-%d %H:%M:%S"
)

print(df["datetime"].dtype)

datetime64[us]


In [8]:
df["event_date"] = df["datetime"].dt.date
df["event_time"] = df["datetime"]

df[["msisdn", "datetime", "event_date", "event_time"]].head()

,msisdn,datetime,event_date,event_time
0,subscribers_00323,2026-02-01 01:23:00,2026-02-01,2026-02-01 01:23:00
1,subscribers_00323,2026-02-01 03:55:00,2026-02-01,2026-02-01 03:55:00
2,subscribers_00323,2026-02-01 06:53:00,2026-02-01,2026-02-01 06:53:00
3,subscribers_00323,2026-02-01 07:52:00,2026-02-01,2026-02-01 07:52:00
4,subscribers_00323,2026-02-01 09:07:00,2026-02-01,2026-02-01 09:07:00


In [9]:
print("Missing event_date:", df["event_date"].isna().sum())
print("Missing event_time:", df["event_time"].isna().sum())

print("Unique event dates:", df["event_date"].nunique())
print("Minimum date:", df["event_date"].min())
print("Maximum date:", df["event_date"].max())

Missing event_date: 0
Missing event_time: 0
Unique event dates: 28
Minimum date: 2026-02-01
Maximum date: 2026-02-28


In [10]:
partition_sizes = (
    df.groupby(["msisdn", "event_date"])
      .size()
)

print("===== Partition Statistics =====")
print(partition_sizes.describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).round(2))

print("\nTotal Partitions:", len(partition_sizes))
print("Largest Partition:", partition_sizes.max())
print("Smallest Partition:", partition_sizes.min())

===== Partition Statistics =====
count    77711.00
mean        13.80
std         19.57
min          1.00
50%         10.00
90%         28.00
95%         33.00
99%         40.00
max        289.00
dtype: float64

Total Partitions: 77711
Largest Partition: 289
Smallest Partition: 1


In [12]:
cell_time_groups = (
    df.groupby(["cell_id", "datetime"])
      .size()
)

duplicate_cell_time = cell_time_groups[
    cell_time_groups > 1
]

print("===== Cell + Timestamp Uniqueness =====")
print("Total groups:", len(cell_time_groups))
print("Duplicate groups:", len(duplicate_cell_time))
print("Records in duplicate groups:", duplicate_cell_time.sum())
print("Maximum events at the same timestamp:", cell_time_groups.max())

===== Cell + Timestamp Uniqueness =====
Total groups: 1071782
Duplicate groups: 826
Records in duplicate groups: 1652
Maximum events at the same timestamp: 2


In [13]:
cell_time_subscriber_groups = (
    df.groupby(["cell_id", "datetime", "msisdn"])
      .size()
)

duplicate_groups = cell_time_subscriber_groups[
    cell_time_subscriber_groups > 1
]

print("Duplicate cell + timestamp + subscriber groups:", len(duplicate_groups))
print("Records in duplicate groups:", duplicate_groups.sum())
print("Maximum events per group:", cell_time_subscriber_groups.max())


Duplicate cell + timestamp + subscriber groups: 0
Records in duplicate groups: 0
Maximum events per group: 1
